# 03 — Team prediction and tournament simulation

This notebook loads the selected probability model produced by Notebook 02 and the frozen team-state snapshot produced by Notebook 01.

It validates team-specific match probabilities, checks neutral-order symmetry and runs the generic 48-team demonstration. The bundled demonstration uses the top 48 teams by frozen Elo and is intentionally separate from the actual-field retrospective analysis in Notebook 04.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.worldcup_runtime import (
    MatchPredictor,
    build_seeded_groups,
    run_monte_carlo_tournament,
)

outputs_dir = project_root / "outputs"
models_dir = project_root / "models"

model_path = models_dir / "selected_probability_model.joblib"
metadata_path = models_dir / "model_metadata.json"
team_state_path = outputs_dir / "team_state_snapshot.csv"

required = [model_path, metadata_path, team_state_path]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required artefacts are missing. Run notebooks 01 and 02 first: "
        + ", ".join(missing)
    )

selected_probability_model = joblib.load(model_path)
model_metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)
team_state_snapshot = pd.read_csv(
    team_state_path,
    parse_dates=["latest_match_date"],
)

predictor = MatchPredictor(
    selected_probability_model,
    team_state_snapshot,
    feature_columns=model_metadata["feature_columns"],
)

selected_probability_model_name = (
    model_metadata["selected_model_name"]
)

print(
    "Loaded model:",
    selected_probability_model_name,
)
print(
    "Frozen team states:",
    len(team_state_snapshot),
)

In [ ]:
example_fixtures = [
    ("Spain", "Brazil"),
    ("France", "Argentina"),
    ("England", "Germany"),
    ("Japan", "South Korea"),
]

prediction_example_rows = []

for home_team, away_team in example_fixtures:
    if (
        home_team not in predictor.available_teams
        or away_team not in predictor.available_teams
    ):
        continue

    prediction = predictor.get_match_probabilities(
        home_team,
        away_team,
        tournament_weight=1.0,
        neutral=True,
    )

    prediction_example_rows.append({
        "home_team": home_team,
        "away_team": away_team,
        "home_elo": float(
            predictor.team_state_lookup.loc[
                home_team,
                "current_elo",
            ]
        ),
        "away_elo": float(
            predictor.team_state_lookup.loc[
                away_team,
                "current_elo",
            ]
        ),
        "home_form_goals_for": float(
            predictor.team_state_lookup.loc[
                home_team,
                "current_form_goals_for",
            ]
        ),
        "away_form_goals_for": float(
            predictor.team_state_lookup.loc[
                away_team,
                "current_form_goals_for",
            ]
        ),
        "home_win_probability": prediction["home_win"],
        "draw_probability": prediction["draw"],
        "away_win_probability": prediction["away_win"],
        "probability_sum": (
            prediction["home_win"]
            + prediction["draw"]
            + prediction["away_win"]
        ),
    })

team_specific_prediction_examples = pd.DataFrame(
    prediction_example_rows
)
team_specific_prediction_examples.to_csv(
    outputs_dir / "team_specific_prediction_examples.csv",
    index=False,
)

if not np.allclose(
    team_specific_prediction_examples["probability_sum"],
    1.0,
    atol=1e-8,
):
    raise ValueError(
        "Team-specific probabilities do not sum to one."
    )

symmetry_rows = []

for home_team, away_team in example_fixtures:
    if (
        home_team not in predictor.available_teams
        or away_team not in predictor.available_teams
    ):
        continue

    forward = predictor.get_match_probabilities(
        home_team,
        away_team,
        1.0,
        True,
    )
    reverse = predictor.get_match_probabilities(
        away_team,
        home_team,
        1.0,
        True,
    )

    symmetry_rows.append({
        "fixture": f"{home_team} vs {away_team}",
        "home_swap_difference": abs(
            forward["home_win"]
            - reverse["away_win"]
        ),
        "away_swap_difference": abs(
            forward["away_win"]
            - reverse["home_win"]
        ),
        "draw_swap_difference": abs(
            forward["draw"]
            - reverse["draw"]
        ),
    })

team_prediction_validation = pd.DataFrame(
    symmetry_rows
)
team_prediction_validation.to_csv(
    outputs_dir / "team_prediction_validation.csv",
    index=False,
)

if (
    team_prediction_validation[[
        "home_swap_difference",
        "away_swap_difference",
        "draw_swap_difference",
    ]].to_numpy().max()
    > 1e-10
):
    raise ValueError(
        "Neutral match predictions are not order-symmetric."
    )

display(team_specific_prediction_examples)

In [ ]:
default_tournament_teams = (
    team_state_snapshot
    .sort_values(
        "current_elo",
        ascending=False,
    )
    .head(48)["team"]
    .tolist()
)

default_groups = build_seeded_groups(
    default_tournament_teams,
    predictor,
    group_count=12,
    group_size=4,
)

group_configuration_rows = []

for group_name, teams in default_groups.items():
    for seed_position, team in enumerate(
        teams,
        start=1,
    ):
        group_configuration_rows.append({
            "group": group_name,
            "seed_position": seed_position,
            "team": team,
            "current_elo": float(
                predictor.team_state_lookup.loc[
                    team,
                    "current_elo",
                ]
            ),
            "selection_note": "top_48_by_current_elo_demo",
        })

tournament_group_configuration = pd.DataFrame(
    group_configuration_rows
)
tournament_group_configuration.to_csv(
    outputs_dir / "tournament_group_configuration.csv",
    index=False,
)

predictor.precompute_neutral_pairs(default_tournament_teams)

simulation_iterations = 2000
simulation_seed = 42

tournament_simulation_results = run_monte_carlo_tournament(
    predictor,
    default_groups,
    iterations=simulation_iterations,
    seed=simulation_seed,
)
tournament_simulation_results.to_csv(
    outputs_dir / "tournament_simulation_results.csv",
    index=False,
)

champion_probability_sum = float(
    tournament_simulation_results[
        "champion_probability"
    ].sum()
)

simulation_validation = pd.DataFrame([
    ("configured_teams", len(default_tournament_teams)),
    ("configured_groups", len(default_groups)),
    ("teams_per_group", 4),
    ("automatic_top_two_qualifiers", 24),
    ("best_third_place_qualifiers", 8),
    ("round_of_32_teams", 32),
    ("simulation_iterations", simulation_iterations),
    ("simulation_seed", simulation_seed),
    ("champion_probability_sum", champion_probability_sum),
    ("cached_probability_matchups", predictor.cache_size),
], columns=["metric", "value"])

simulation_validation.to_csv(
    outputs_dir / "tournament_simulation_validation.csv",
    index=False,
)

if not np.isclose(
    champion_probability_sum,
    1.0,
    atol=1e-9,
):
    raise ValueError(
        "Champion probabilities do not sum to one."
    )

figures_dir = outputs_dir / "figures"
figures_dir.mkdir(exist_ok=True)

top_champions = tournament_simulation_results.head(15)

plt.figure(figsize=(9, 6))
plt.barh(
    top_champions["team"][::-1],
    top_champions[
        "champion_probability"
    ][::-1] * 100,
)
plt.xlabel("Champion probability (%)")
plt.ylabel("Team")
plt.title(
    f"Top simulated champions ({simulation_iterations:,} iterations)"
)
plt.tight_layout()
plt.savefig(
    figures_dir / "champion_probabilities.png",
    dpi=150,
    bbox_inches="tight",
)
plt.close()

final_run_validation = pd.DataFrame([
    ("training_matches", model_metadata["training_rows"]),
    ("test_matches", model_metadata["test_rows"]),
    ("historical_holdout_end", model_metadata["holdout_end"]),
    ("selected_probability_model", selected_probability_model_name),
    ("team_state_count", len(team_state_snapshot)),
    ("tournament_demo_teams", len(default_tournament_teams)),
    ("simulation_iterations", simulation_iterations),
    ("simulation_seed", simulation_seed),
    ("champion_probability_sum", champion_probability_sum),
], columns=["metric", "value"])

final_run_validation.to_csv(
    outputs_dir / "final_run_validation.csv",
    index=False,
)

print(
    "Selected probability model:",
    selected_probability_model_name,
)
display(
    tournament_simulation_results.head(15)
)

## Interpretation

The generic tournament is a modelling demonstration. Group ties use points followed by frozen Elo because the match model does not predict scorelines, and the knockout pairing is a transparent seeded approximation rather than the official FIFA bracket.